Use Hugging Face and SBERT plaforms
Sometimes the results may differ, the logic of the solution is taken into account
Use this input:
Polish = ["Czuję się niesamowicie szczęśliwy, bo świeci słońce.",
"To jest najgorszy dzień w moim życiu—zgubiłem portfel.",
"Nie mogę przestać się śmiać z tego żartu!",
"Film był tak inspirujący, że czuję się zmotywowany.",
"Jestem taki dumny ze swoich osiągnięć.",
"Czuję się tak samotny bez moich przyjaciół.",
"Ten projekt doprowadza mnie do szału.",
"Byłem wzruszony jej dobrocią.",
"Jestem podekscytowany nadchodzącą podróżą.",
"Nie mam żadnego przyjaciela i bardzo mi smutno z tego powodu."]


In [25]:
!pip install sentence-transformers
!pip install transformers
!pip install sentencepiece
!pip install sacremoses
!pip install huggingface_hub
!huggingface-cli login

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 9.9 MB/s eta 0:00:00

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `huggingface-cli whoami` to get more information or `huggingface-cli logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be v

In [57]:
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification, pipeline

In [52]:
polish_text = ["Czuję się niesamowicie szczęśliwy, bo świeci słońce.", "To jest najgorszy dzień w moim życiu—zgubiłem portfel.", "Nie mogę przestać się śmiać z tego żartu!", "Film był tak inspirujący, że czuję się zmotywowany.", "Jestem taki dumny ze swoich osiągnięć.", "Czuję się tak samotny bez moich przyjaciół.", "Ten projekt doprowadza mnie do szału.", "Byłem wzruszony jej dobrocią.", "Jestem podekscytowany nadchodzącą podróżą.", "Nie mam żadnego przyjaciela i bardzo mi smutno z tego powodu."]

1. Translate the given sentences into English (1 point)
Tip: use “translation” Hugging Face pipeline and find some translation model
here https://huggingface.co/models?pipeline_tag=translation


In [60]:
model_name = "Helsinki-NLP/opus-mt-pl-en"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
inputs = tokenizer(polish_text, return_tensors="pt", padding=True, truncation=True)
outputs = model.generate(**inputs, max_length=50, num_beams=4, early_stopping=True)
translations = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
print("Translations:", translations)

Translations: ['I feel incredibly happy because the sun is shining.', "This is the worst day of my life I've lost my wallet.", "I can't stop laughing at this joke!", 'The movie was so inspiring, I feel motivated.', "I'm so proud of my accomplishments.", 'I feel so alone without my friends.', 'This project is driving me crazy.', 'I was touched by her kindness.', "I'm excited to be on the road.", "I don't have a friend, and I'm very sad about it."]


2. Classify the emotions of each English sentence (1 point)
Tip: use “text-classification” Hugging Face pipeline and i.e. “distilbert-baseuncased-finetuned-sst-2-english" model
Example output:
I feel incredibly happy because the sun is shining.: POSITIVE
This is the worst day of my life I've lost my wallet.: NEGATIVE
I can't stop laughing at this joke!: POSITIVE
The movie was so inspiring, I feel motivated.: POSITIVE
I'm so proud of my accomplishments.: POSITIVE
I feel so alone without my friends.: NEGATIVE
This project is driving me crazy.: POSITIVE
I was touched by her kindness.: POSITIVE
I'm excited to be on the road.: POSITIVE
I don't have a friend, and I'm really sad about it.: NEGATIVE

In [35]:

tokenizer_emo = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")
model = AutoModelForSequenceClassification.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")
for translation in translations:
  inputs = tokenizer_emo(translation, return_tensors="pt")
  outputs = model(**inputs)
  predicted_class_id = outputs.logits.argmax().item()
  predicted_class = model.config.id2label[predicted_class_id]
  print(f"{translation}: {predicted_class}")


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


I feel incredibly happy because the sun is shining.: positive
This is the worst day of my life I've lost my wallet.: negative
I can't stop laughing at this joke!: positive
The movie was so inspiring, I feel motivated.: positive
I'm so proud of my accomplishments.: positive
I feel so alone without my friends.: negative
This project is driving me crazy.: negative
I was touched by her kindness.: positive
I'm excited to be on the road.: positive
I don't have a friend, and I'm very sad about it.: negative


3. Expand translated sentences using text generation (1 point)
Tip: use "text-generation" Hugging Face pipeline and gpt2 model
Example output:
{"I feel incredibly happy because the sun is shining. For someone who is sick at
heart, a bad day doesn't mean we'll always fall victim to"}
{"This is the worst day of my life I've lost my wallet. I'd like my card number to
know if someone else does. It's hard"}
{"I can't stop laughing at this joke! What a joke! All of the fans in the stadium
booed him… The Red Bull has to pay"}
{"The movie was so inspiring, I feel motivated. A lot of other people feel like
they're just reading the script and they're waiting for the director"}
{'I\'m so proud of my accomplishments.\n\n"If anyone\'s looking for that big
chance to take me down in a game, I\'ll tell'}
{"I feel so alone without my friends. But my friends are mine.\n\nWhat's so
wonderful about that?\n\nWell, I get to"}
{"This project is driving me crazy. This is a project that is absolutely ridiculous,
and I have to admit, a joke, and it doesn't work"}
{"I was touched by her kindness. I didn't think if I knew why she took so much
time for her. But now it's gone. I"}
{"I'm excited to be on the road. It's been such a long road and one of the hardest
things that I've done. I've always"}
{"I don't have a friend, and I'm really sad about it. I don't know what to do. The
thing I've always liked about"}

In [47]:
generator = pipeline("text-generation", model="gpt2")
expanded_sentences = generator(translations, max_length=50, num_return_sequences=1)
for idx, sentence in enumerate(translations):
    print(f"Original Sentence {idx + 1}: {sentence}")
    expanded_sentences = generator(sentence, max_length=50, num_return_sequences=1)
    for result in expanded_sentences:
        print(f"Expansion: {result['generated_text']}")
    print()

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Setting `pad_token_id` to `eos_token_id`:None for open-end 

Original Sentence 1: I feel incredibly happy because the sun is shining.


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Expansion: I feel incredibly happy because the sun is shining. It was just me who started getting a second tattoo, because there was no other option.

In my childhood you saw people with tattoos everywhere all over their faces, you'd see it on the

Original Sentence 2: This is the worst day of my life I've lost my wallet.


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


...

Expansion: This project is driving me crazy. It's going to be cool to learn a new way to be able to sit down with a game creator and understand how I do it."

A few months ago, they were still working on their own game

Original Sentence 8: I was touched by her kindness.


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Expansion: I was touched by her kindness. There was an element of empathy there that I didn't think I'd ever get," added the 23-year-old senior, who has had the opportunity to take part in the event despite having been "wounded

Original Sentence 9: I'm excited to be on the road.


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Expansion: I'm excited to be on the road. I would say I'm still excited about the sport and what I'm doing. It'll be fun."

The goal is to develop people and build an organization. That's the goal. It depends

Original Sentence 10: I don't have a friend, and I'm very sad about it.
Expansion: I don't have a friend, and I'm very sad about it. I think it's really sad."

Read more:

A couple in Alabama say they feel bad because they're gay when he said there are no legal protections against



4. Recognize the language for the first sentence in Polish and translated into
English (1 point)
Tip: use “text-classification” Hugging Face pipeline and i.e. “papluca/xlmroberta-base-language-detection" model
Expected Output:
Czuję się niesamowicie szczęśliwy, bo świeci słońce. : [{'label': 'pl', 'score':
0.9945397973060608}]
I feel incredibly happy because the sun is shining. : [{'label': 'en', 'score':
0.9935412406921387}]


In [54]:
model_name = "nie3e/gpt2-lang-ident"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
for sentence in polish_text:
  inputs = tokenizer(sentence, return_tensors="pt")
  outputs = model(**inputs)
  predicted_class_id = outputs.logits.argmax().item()
  predicted_class = model.config.id2label[predicted_class_id]
  print(f"{sentence}: {predicted_class}")


Czuję się niesamowicie szczęśliwy, bo świeci słońce.: pl
To jest najgorszy dzień w moim życiu—zgubiłem portfel.: pl
Nie mogę przestać się śmiać z tego żartu!: pl
Film był tak inspirujący, że czuję się zmotywowany.: pl
Jestem taki dumny ze swoich osiągnięć.: pl
Czuję się tak samotny bez moich przyjaciół.: pl
Ten projekt doprowadza mnie do szału.: pl
Byłem wzruszony jej dobrocią.: pl
Jestem podekscytowany nadchodzącą podróżą.: pl
Nie mam żadnego przyjaciela i bardzo mi smutno z tego powodu.: pl


5. Analyze the similarity between the sentences translated into English (1
point)
Tip: use SBERT (SentenceTransformer) SentenceTransformer('sentencetransformers/all-MiniLM-L6-v2')
Expected Output:
(('I feel so alone without my friends.', "I don't have a friend, and I'm really sad
about it."), 0.5773568153381348, ("I can't stop laughing at this joke!", 'I feel so
alone without my friends.'), 0.009623957797884941)

In [64]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
sentence_embeddings = model.encode(translations)
similar_sentences = []
for i in range(len(translations)):
    for j in range(i + 1, len(translations)):
        # Use cosine similarity from the util module
        similarity = util.cos_sim(sentence_embeddings[i], sentence_embeddings[j]).item()
        similar_sentences.append((translations[i], translations[j], similarity))

# Sort by similarity in descending order
similar_sentences.sort(key=lambda x: x[2], reverse=True)

# Print top 4 most similar pairs
for sentence1, sentence2, similarity in similar_sentences[:4]:
    print(f"({sentence1}, {sentence2}) -> Similarity: {similarity:.4f}")


(I feel so alone without my friends., I don't have a friend, and I'm very sad about it.) -> Similarity: 0.5575
(I feel incredibly happy because the sun is shining., I'm excited to be on the road.) -> Similarity: 0.3524
(I feel incredibly happy because the sun is shining., The movie was so inspiring, I feel motivated.) -> Similarity: 0.3478
(The movie was so inspiring, I feel motivated., I'm so proud of my accomplishments.) -> Similarity: 0.3297
